# MNIST Digits Recognition - Data acquisition 

### Configure proxies if needed

In [1]:
# Specify proxy and no_proxy value if applicable
http_proxy = "" ## "http://<proxy_host>:<port>"
# After default.svc append comma separated list of IP address, hostname, CIDR block, service name, etc.
no_proxy = "" ## "localhost,.cluster.local,.svc,.default.svc, <host>:<port>, x.x.x.x/y,..."

### Copy required directoryies from share to work directory

In [2]:
mnist_work_dir = "MNIST-Digits-Recognition"

In [3]:
#!cp -r ~/shared/ezua-tutorials/current-release/Data-Science/Kubeflow/MNIST-Digits-Recognition/dataset ~/{mnist_work_dir}
!cp -r ~/shared/aie-tutorials/current-release/Data-Science/Kubeflow/MNIST-Digits-Recognition/dataset ~/{mnist_work_dir}

In [4]:
#!cp -r ~/shared/ezua-tutorials/current-release/Data-Science/Kubeflow/MNIST-Digits-Recognition/image ~/{mnist_work_dir}
!cp -r ~/shared/aie-tutorials/current-release/Data-Science/Kubeflow/MNIST-Digits-Recognition/image ~/{mnist_work_dir}

### Create and populate S3 bucket

In [5]:
# Code to copy mnist digit dataset from ezua-tutorials to ezaf-demo bucket at data/mnist for Digit Recognition Example
import os, boto3
s3 = boto3.client("s3", verify=False)
bucket = 'ezaf-demo'
source_dir = '/mnt/shared/aie-tutorials/current-release/Data-Science/Kubeflow/MNIST-Digits-Recognition/dataset'
dest_dir = 'data/mnist'
# Get list of files under dataset dir
dataset_list = os.listdir(source_dir)# Create source file path and destination object key strings
source_files = []
dest_objects = []
for dataset in dataset_list:
    source_files.append(source_dir + '/' + dataset)
    dest_objects.append(dest_dir + '/' + dataset)# check whether bucket is already created
buckets = s3.list_buckets()
bucket_exists = False
available_buckets = buckets["Buckets"]
for available_bucket in available_buckets:
    if available_bucket["Name"] == bucket:
        bucket_exists = True
        break

Found endpoint for s3 via: environment_global.


In [6]:
if not bucket_exists:
    s3.create_bucket(Bucket=bucket)# Upload files
for i in range(len(source_files)):
    s3.upload_file(Filename=source_files[i], Bucket=bucket, Key=dest_objects[i]) 

In [7]:
# s3.list_buckets()

In [8]:
# s3.list_objects_v2(Bucket='ezaf-demo')

In [9]:
!cp ~/shared/aie-tutorials/current-release/Data-Analytics/Spark/object_store_secret.yaml.tpl ~

### Generate a secret to read data source files from S3 bucket by Spark application

In [10]:
import kfp
kfp_client = kfp.Client()
namespace = kfp_client.get_user_namespace()

/opt/conda/lib/python3.11/site-packages/kfp/client/client.py:159: FutureWarning: This client only works with Kubeflow Pipeline v2.0.0-beta.2 and later versions.
  warnings.warn(


In [11]:
namespace

'ashish-kumarj-h-653cd6af'

In [12]:
!sed -e "s/\$AUTH_TOKEN/$AUTH_TOKEN/" ~/object_store_secret.yaml.tpl > ~object_store_secret.yaml

In [13]:
!kubectl apply -f ~object_store_secret.yaml -n {namespace}

secret/spark-s3-creds created


## Run Airflow DAG

**NOTE**: This task has to be performed manually. 

Replace the following variable with your PCAI's Ezmeral AI Essential main link:

In [36]:
ezaie_url = "https://home.prod.discover.hpepcai.com"

In [37]:
airflow_url = ezaie_url.replace("home", "airflow")
airflow_url

'https://airflow.prod.discover.hpepcai.com'

Access Airflow at the URL above, and run the following job:   
**spark_read_write_parquet_mnist**

### Check that data has been processed by Spark and Parquet files are present in shared/mnist-spark-data directory:

In [14]:
!ls ~/shared/mnist-spark-data/

part-00000-9203b8b2-3e68-48b2-8d25-bbb394145d17-c000.snappy.parquet  _SUCCESS
part-00001-9203b8b2-3e68-48b2-8d25-bbb394145d17-c000.snappy.parquet


In [15]:
# Specify the name of the user mounted directory
user_mounted_dir_name = namespace

### Copy Parquet data from shared directory to local directory

In [16]:
!cp -r ~/shared/mnist-spark-data ~/{user_mounted_dir_name}

Now you should have the Spark-processed parquet files into the mnist-spark-data inside your user volume:

In [17]:
!ls ~/{user_mounted_dir_name}/mnist-spark-data

part-00000-9203b8b2-3e68-48b2-8d25-bbb394145d17-c000.snappy.parquet  _SUCCESS
part-00001-9203b8b2-3e68-48b2-8d25-bbb394145d17-c000.snappy.parquet


### Read Parquet files and transform them into Pickle files in local directory

In [18]:
from pathlib import Path

# The source training data contains Parquet files written by the Spark job
source_training_dir = f"{str(Path.home())}/{user_mounted_dir_name}/mnist-spark-data"

# The target training data is obtained by transforming the source training data from idx format to npy format
target_training_dir = f"{str(Path.home())}/{user_mounted_dir_name}/training"

In [19]:
#!ls {target_training_dir}
#source_training_dir, target_training_dir

In [20]:
def pip_install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

In [21]:
pip_install("idx2numpy")

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for idx2numpy: filename=idx2numpy-1.2.3-py3-none-any.whl size=8002 sha256=7ad387c0f61cbd9cf87bb4b97ff6aa675aa516d50a0c0739483255f13ce039dd
  Stored in directory: /home/ashish-kumarj-hpe.com/.cache/pip/wheels/87/e5/e7/70fc742b3645ddf9d392f766feccbcc95cb3a3c806f8588af0
Successfully built idx2numpy


### Transformation of the training data from Apache Parquet->IDX format into Numpy format required for the MNIST example

In [22]:
# Transformation of the training data from Apache Parquet format to the format that is required for the MNIST example
import pandas as pd
import numpy as np
import gzip
import idx2numpy
import os

if not os.path.exists(target_training_dir):
    os.makedirs(target_training_dir)
    
mnist_parquet = pd.read_parquet(source_training_dir)
x_train_gz, x_test_gz, y_train_gz, y_test_gz = mnist_parquet["content"]

x_train_npy = idx2numpy.convert_from_string(gzip.decompress(x_train_gz))
np.save(target_training_dir + "/train-images-pickle.npy", x_train_npy, allow_pickle=True)

x_test_npy = idx2numpy.convert_from_string(gzip.decompress(x_test_gz))
np.save(target_training_dir + "/test-images-pickle.npy", x_test_npy, allow_pickle=True)

y_train_npy = idx2numpy.convert_from_string(gzip.decompress(y_train_gz))
np.save(target_training_dir + "/train-labels-pickle.npy", y_train_npy, allow_pickle=True)

y_test_npy = idx2numpy.convert_from_string(gzip.decompress(y_test_gz))
np.save(target_training_dir + "/test-labels-pickle.npy", y_test_npy, allow_pickle=True)

In [23]:
# test pickle
#x_train = np.load(target_training_dir + "/train-images-pickle.npy", allow_pickle=True) 

#### Check the Pickle files are present in the local training directory

In [24]:
!ls {target_training_dir}/*.npy

/home/ashish-kumarj-hpe.com/ashish-kumarj-h-653cd6af/training/test-images-pickle.npy
/home/ashish-kumarj-hpe.com/ashish-kumarj-h-653cd6af/training/test-labels-pickle.npy
/home/ashish-kumarj-hpe.com/ashish-kumarj-h-653cd6af/training/train-images-pickle.npy
/home/ashish-kumarj-hpe.com/ashish-kumarj-h-653cd6af/training/train-labels-pickle.npy


## SHORTCUT - Alternative way to create the pickle files

All the above cells perform acquisition and transformation steps using Airflow+Spark.   
In this specific MNIST use case, you may want to skip those steps and directly create the pickle files, and directly go the MLOps pipeline.   
In this case, uncomment the cell below to create the very same files using Keras library. To uncomment the cell, select the cell's content and press **CTRL** + **/**

In [75]:
# import kfp
# import keras

# kfp_client = kfp.Client()
# namespace = kfp_client.get_user_namespace()

# target_training_dir = f"{str(Path.home())}/{user_mounted_dir_name}/training"

# if not os.path.exists(target_training_dir):
#     os.makedirs(target_training_dir)

# (train_images, train_labels), (test_images, test_labels) = keras.datasets.mnist.load_data()
# np.save(f"{target_training_dir}/train-images-pickle.npy", train_images)
# np.save(f"{target_training_dir}/train-labels-pickle.npy", train_labels)
# np.save(f"{target_training_dir}/test-images-pickle.npy", test_images)
# np.save(f"{target_training_dir}/test-labels-pickle.npy", test_labels)

## Create K8s storage volume for exporting model 
#### This is a prerequisite for running the following MLOps pipeline using Kale

In [76]:
import os
import uuid
import kfp
import kfp.dsl as dsl
from kfp import components

volume_str = "model-volume-kale"

# Create volume to serve the model.
!sed -e "s/pvc-name/$volume_str/" ./pvc.yaml > /tmp/pvc.yaml
!kubectl apply -f /tmp/pvc.yaml

persistentvolumeclaim/model-volume-kale unchanged


### Next step: MLOps pipeline

Go the the following step:  
[Simple MLOps pipeline notebook](02-MLOps-pipeline-kfp.ipynb)   
or   
[MLOps pipeline notebook with hyperparameter optimization](02-MLOps-pipeline-HPOptimization.ipynb)